In [5]:
# HW 9 Problem 2 with neural net 
# !pip install torch
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
sns.set()

# bring in the data set
df=pd.read_csv("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv")

def fit_predict(model_name,Xtrn,Ytrn,Xtest,seed=0,itrn=3000,lr=1e-2):
    if model_name=='linear':
        return LinearRegression().fit(Xtrn,Ytrn.predict(Xtest))
    torch.manual_seed(seed)
    #normalize the data 
    sx = StandardScaler().fit(Xtrn)
    sy = StandardScaler().fit(Ytrn.reshape(-1,1))
    #convert to pytorch tensors
    xtrn = torch.from_numpy(sx.transform(Xtrn)).float()
    ytrn = torch.from_numpy(sy.transform(Ytrn.reshape(-1,1))).float()
    xtest = torch.from_numpy(sx.transform(Xtest)).float()

    if model_name == 'nn_2': #2 hidden nodes
        model = torch.nn.Sequential(
            toch.nn.Linear(5,2),
            torch.nn.ReLU(),
            torch.nn.Linear(2,1),)
    else: #4 hidden nodes over 2 layers
        model = torch.nn.Sequential( 
            torch.nn.Linear(5, 2),
            torch.nn.ReLU(),
            torch.nn.Linear(2, 2),
            torch.nn.ReLU(),
            torch.nn.Linear(2, 1),)
    loss_fn = torch.nn.MSELoss() #means squared error loss
    opt = torch.optim.Adam(model.parameters(),lr=lr) 

    for t in range(itrn):
        opt.zero_grad() #zero gradients 
        loss = loss_fn(model(xtrn),ytrn) #predict on training data and find error
        loss.backward() #backward propogation
        opt.step() #optimize weights

        if t % 500==0:
            print(model_name,t,loss.item())

    with torch.no_grad():
        pred = model(xtest),numpy()
    return sy.inverse_transform(pred).ravel()
                                      
#check this
XA, YA = make_lagged('Brazil')
XB, YB = make_lagged('Peru')
h = len(YA) // 2

scenarios = {
    'a: all A -> all A':      (XA, YA, XA, YA),
    'b: 1st half -> 2nd half': (XA[:h], YA[:h], XA[h:], YA[h:]),
    'c: A -> B':               (XA, YA, XB, YB),
}

mse, errors = {}, {}
for sname, (Xtr, Ytr, Xte, Yte) in scenarios.items():
    for mname in ['linear', 'nn_2', 'nn_2x2']:
        err = Yte - fit_predict(mname, Xtr, Ytr, Xte)
        errors[(sname, mname)] = err
        mse[(sname, mname)] = np.mean(err**2)

table = pd.Series(mse).unstack()      # rows = scenarios, columns = models
print(table.applymap(lambda v: f"{v:.3g}"))
                                  